<a href="https://colab.research.google.com/github/1kaiser/opyf_colab/blob/main/pipeline_nb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# opyf_colab — Hydraulic Analysis Pipeline

**Brague flood event reconstruction** — monocular depth · JAX LSPIV ·
D8 thalweg · IS 10430 canal design.

Run all cells top-to-bottom, or via Papermill:
```bash
JAX_PLATFORMS=cpu conda run -n num_gpu papermill pipeline_nb.ipynb pipeline_nb.ipynb \
  -p SKIP_DOWNLOAD True -p SKIP_DEPTH True -p SKIP_ALIGN True
```

In [7]:
!git clone https://github.com/1kaiser/opyf_colab

fatal: destination path 'opyf_colab' already exists and is not an empty directory.


In [8]:
%cd opyf_colab

/content/opyf_colab


In [22]:
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/IMG_1139.MOV -O /content/opyf_colab/data/brague/IMG_1139.MOV
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/IMG_1142.MOV -O /content/opyf_colab/data/brague/IMG_1142.MOV
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/MNT.xyz -O /content/opyf_colab/data/brague/MNT.xyz
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/Ortho.tif -O /content/opyf_colab/data/brague/Ortho.tif
!mkdir -p /content/opyf_colab/weights
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/depth_pro.msgpack -O /content/opyf_colab/weights/depth_pro.msgpack
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/superpoint.msgpack -O /content/opyf_colab/weights/superpoint.msgpack
!wget -q https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/superpoint_lightglue.msgpack -O /content/opyf_colab/weights/superpoint_lightglue.msgpack


In [23]:
# Papermill parameters — override any value from the CLI with -p KEY VALUE

VIDEO_DOWN    = "data/brague/IMG_1139.MOV"
VIDEO_UP      = "data/brague/IMG_1142.MOV"
MNT_XYZ       = "data/brague/MNT.xyz"
ORTHO_TIF     = "data/brague/Ortho.tif"
DP_WEIGHTS    = "weights/depth_pro.msgpack"
SP_WEIGHTS    = "weights/superpoint.msgpack"
LG_WEIGHTS    = "weights/superpoint_lightglue.msgpack"
OUT_DIR       = "output/brague"
ASSETS_DIR    = "assets"
N_FRAMES      = 5

SKIP_DOWNLOAD  = True
SKIP_DEPTH     = True
SKIP_ALIGN     = True
SKIP_D8        = False
SKIP_PC_CHECK  = False
SKIP_CANAL     = False
SKIP_CANAL_VIZ = False
SKIP_LSPIV     = False
SKIP_VIZ       = False

In [24]:
# Parameters
SKIP_DOWNLOAD = True
SKIP_DEPTH = True
SKIP_ALIGN = True


In [25]:
import os, sys, json, types, time
from pathlib import Path

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.chdir(Path(__file__).parent if "__file__" in dir() else Path("."))

print("Working dir :", Path(".").resolve())
print("Python      :", sys.executable)

Working dir : /content/opyf_colab
Python      : /usr/bin/python3


## Stage 0 — Download release assets
Downloads videos, weights, MNT, Ortho from the GitHub release if absent.

In [26]:
if not SKIP_DOWNLOAD:
    from pipeline import download_assets
    download_assets()
else:
    print("Stage 0 skipped — assets assumed present")

Stage 0 skipped — assets assumed present


## Stages 1–5 — Depth inference pipeline

Extracts frames → Depth Pro inference → GCP-aligned metric depth →
`flow_depth.tif`  (aggregated median over N frames).

In [ ]:
meta_path = Path(OUT_DIR) / "pipeline_meta.json"
flow_tif  = Path(OUT_DIR) / "flow_depth.tif"

if SKIP_DEPTH and flow_tif.exists() and meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print("Depth pipeline skipped — loaded", meta_path)
    print(f"  h_mean={meta['h_final_mean']:.3f} m   h_max={meta['h_final_max']:.3f} m")

    # generate frame_sample + flow_depth_result from cached tifs
    import shutil, numpy as np, rasterio as _rio
    frames_dir = Path(OUT_DIR) / "frames"
    existing = sorted(frames_dir.glob("*.png")) if frames_dir.exists() else []
    if existing:
        Path(ASSETS_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copy(existing[0], Path(ASSETS_DIR) / "frame_sample.png")
        print("  frame_sample →", Path(ASSETS_DIR) / "frame_sample.png")

    from pipeline import _save_flow_depth_result
    from modules.depth_to_elevation import load_mnt, rasterise_mnt
    z_surf_tifs = sorted(Path(OUT_DIR).glob("*_z_surface.tif"))
    if flow_tif.exists() and z_surf_tifs:
        with _rio.open(ORTHO_TIF) as src:
            _tf = src.transform; _sh = (src.height, src.width)
        with _rio.open(flow_tif) as src:
            h_arr = src.read(1).astype(np.float32)
        Xm, Ym, Zm = load_mnt(MNT_XYZ, subsample=5)
        zb = rasterise_mnt(Xm, Ym, Zm, _tf, _sh)
        _save_flow_depth_result(h_arr, zb, _sh, Path(ASSETS_DIR))
else:
    args = types.SimpleNamespace(
        video=VIDEO_DOWN, mnt=MNT_XYZ, ortho=ORTHO_TIF,
        weights=DP_WEIGHTS, out_dir=OUT_DIR, assets=ASSETS_DIR,
        n_frames=N_FRAMES, sp_weights=SP_WEIGHTS, lg_weights=LG_WEIGHTS,
    )
    from pipeline import run_depth_pipeline
    meta = run_depth_pipeline(args)


  STAGE 1: Extract event frames


Extracting frames: 100%|██████████| 5/5 [00:03<00:00,  1.33it/s]


  Extracted 5 frames → output/brague/frames
  Frame sample → /content/opyf_colab/assets/frame_sample.png

  STAGE 2: Load Depth Pro
  Loading Depth Pro weights from weights/depth_pro.msgpack ...


## Stage 6b — Point cloud alignment + water volume
LightGlue bank-feature matching → homography → inundated volume (m³).

In [ ]:
if not SKIP_ALIGN:
    import numpy as np, rasterio as _rio
    from modules.depth_to_elevation import load_mnt, rasterise_mnt
    from pipeline import run_alignment_stage

    with _rio.open(ORTHO_TIF) as src:
        _tf    = src.transform
        _shape = (src.height, src.width)
    X_m, Y_m, Z_m = load_mnt(MNT_XYZ, subsample=5)
    z_bed = rasterise_mnt(X_m, Y_m, Z_m, _tf, _shape)

    ar = run_alignment_stage(
        Path(OUT_DIR), z_bed, _tf, X_m, Y_m, Z_m,
        Path(ASSETS_DIR), SP_WEIGHTS, LG_WEIGHTS,
    )
    if ar:
        print(f"Water volume  : {ar['volume_m3']:,.0f} m³")
        print(f"Inundated area: {ar['area_m2']:,.0f} m²")
else:
    print("Stage 6b skipped")

## Stage 6c — JAX D8 thalweg
Fill sinks → D8 flow directions → accumulation → thalweg trace →
slope / curvature / bearing profile.
Output: `assets/d8_thalweg.png`

In [ ]:
d8_result = None
if not SKIP_D8:
    from pipeline import run_d8_thalweg
    d8_result = run_d8_thalweg(Path(OUT_DIR), Path(ASSETS_DIR))
    if d8_result:
        g = d8_result["geometry"]
        print(f"Thalweg  L={g['reach_length_m']:.1f} m   S={g['S_long']:.5f}   "
              f"R_min={g['min_radius_m']:.1f} m")
else:
    print("Stage 6c skipped")

In [ ]:
from IPython.display import Image, display
for f in ["d8_thalweg.png"]:
    p = Path(ASSETS_DIR) / f
    if p.exists(): display(Image(str(p), width=700))

## Stage 6d — Pointcloud × Ortho alignment check
Drapes Ortho.tif RGB onto MNT.xyz elevation; 4-panel diagnostic.
Output: `assets/pointcloud_ortho_check.png`

In [ ]:
if not SKIP_PC_CHECK:
    from pipeline import run_pc_ortho_check
    run_pc_ortho_check(MNT_XYZ, ORTHO_TIF, Path(ASSETS_DIR))
else:
    print("Stage 6d skipped")

In [ ]:
p = Path(ASSETS_DIR) / "pointcloud_ortho_check.png"
if p.exists(): display(Image(str(p), width=900))

## Stage 7 — JAX canal optimiser (IS 10430)
Manning-Strickler minimum wetted-perimeter section, IS velocity compliance.
Output: `canal_design/canal_params.json`,
`assets/canal_section.png`, `assets/design_chain.png`

In [ ]:
canal_dir = Path("canal_design")
cp_path   = canal_dir / "canal_params.json"

if SKIP_CANAL and cp_path.exists():
    with open(cp_path) as f:
        canal_params = json.load(f)
    print("Canal optimizer skipped — loaded", cp_path)
else:
    from pipeline import run_canal_optimizer
    canal_params = run_canal_optimizer(meta, canal_dir, Path(ASSETS_DIR))

print(f"B={canal_params['bed_width_m']:.3f} m  D={canal_params['water_depth_m']:.3f} m  "
      f"Q={canal_params['Q_calculated_m3s']:.2f} m³/s  V={canal_params['velocity_ms']:.3f} m/s")

In [ ]:
for f in ["canal_section.png", "design_chain.png"]:
    p = Path(ASSETS_DIR) / f
    if p.exists(): display(Image(str(p), width=700))

## Stage 7b — Canal 3D overlay
Designed section rendered over reconstructed terrain + ortho background,
D8 thalweg overlaid. Output: `assets/canal_3d_overlay.png`

In [ ]:
reach_geo = None
if not SKIP_CANAL_VIZ:
    from pipeline import run_canal_3d_viz
    reach_geo = run_canal_3d_viz(
        canal_params, Path(OUT_DIR), ORTHO_TIF, Path(ASSETS_DIR), d8_result,
    )
else:
    print("Stage 7b skipped")

In [ ]:
p = Path(ASSETS_DIR) / "canal_3d_overlay.png"
if p.exists(): display(Image(str(p), width=900))

## Stage 7c — Canal 4-view CAD drawing
Front / Side / Top / Isometric with dimension lines (IS 10430).
Output: `assets/canal_cad_model.png`

In [ ]:
if not SKIP_CANAL_VIZ:
    from pipeline import run_canal_cad
    run_canal_cad(canal_params, reach_geo, Path(ASSETS_DIR))
else:
    print("Stage 7c skipped")

In [ ]:
p = Path(ASSETS_DIR) / "canal_cad_model.png"
if p.exists(): display(Image(str(p), width=900))

## Stage 7d — JAX LSPIV surface velocity + discharge
DLT orthorectification → FFT phase-correlation PIV → Gaussian RBF
interpolation → Q = α ∫ V·h dl (α=0.9).

Processes both bridges (IMG_1139 downstream + IMG_1142 upstream).

Also generates three **opyflow-equivalent figures**:

| opyflow original | JAX equivalent |
|---|---|
| `birdEyeTransf1139.png` | `assets/opyflow_birdeye.png` |
| `1139.png` + `1142.png` | `assets/opyflow_velocity_field.png` |
| `figure_Brague.png` | `assets/figure_brague.png` |

In [ ]:
lspiv_result = None
if not SKIP_LSPIV:
    from pipeline import run_lspiv
    lspiv_result = run_lspiv(
        video_down=VIDEO_DOWN,
        video_up=VIDEO_UP,
        mnt_xyz=MNT_XYZ,
        ortho_tif=ORTHO_TIF,
        out_dir=Path(OUT_DIR),
        assets=Path(ASSETS_DIR),
    )
    if lspiv_result:
        print(f"Combined Q = {lspiv_result['Q']:.2f} m³/s")
else:
    print("Stage 7d skipped")

In [ ]:
for f in ["lspiv_results.png", "opyflow_birdeye.png",
          "opyflow_velocity_field.png", "figure_brague.png"]:
    p = Path(ASSETS_DIR) / f
    if p.exists():
        print(f)
        display(Image(str(p), width=900))

## Stage 8 — Annotated pipeline visualisation + future roadmap
Eight-panel summary figure covering the full pipeline.
Output: `assets/annotated_pipeline.png`, `assets/future_roadmap.png`

In [ ]:
if not SKIP_VIZ:
    from pipeline import run_visualisation
    run_visualisation(Path(ASSETS_DIR))
else:
    print("Stage 8 skipped")

In [ ]:
for f in ["annotated_pipeline.png", "future_roadmap.png"]:
    p = Path(ASSETS_DIR) / f
    if p.exists():
        print(f)
        display(Image(str(p), width=900))

## Results summary

In [ ]:
print("=" * 62)
print("  opyf_colab — Brague Flood 2019 · Key Results")
print("=" * 62)
print(f"  Flow depth h_mean        = {meta.get('h_final_mean', 0):.3f} m")
print(f"  Flow depth h_max         = {meta.get('h_final_max', 0):.3f} m")
if d8_result:
    g = d8_result["geometry"]
    print(f"  Thalweg length           = {g['reach_length_m']:.1f} m")
    print(f"  Thalweg slope            = {g['S_long']:.5f}")
print(f"  Canal bed width B        = {canal_params['bed_width_m']:.3f} m")
print(f"  Canal water depth D      = {canal_params['water_depth_m']:.3f} m")
print(f"  Canal Q (IS 10430)       = {canal_params['Q_calculated_m3s']:.2f} m³/s")
if lspiv_result:
    print(f"  LSPIV discharge Q        = {lspiv_result['Q']:.2f} m³/s")
    print(f"  opyflow paper reference  = 102 ± 20 m³/s")
print("=" * 62)